# TP2. Búsqueda y clasificación de razas

Comparación de embeddings, entrenamiento supervisado y clasificación de recortes de YOLO.

## 1. Directorio del proyecto

El kernel parte de la raíz del repositorio o de la ruta indicada por `TP2_ROOT`.

In [ ]:
import os
import sys
from pathlib import Path
# En Colab, completar con la carpeta del fork ya clonado o extraído.
ROOT = Path(os.environ.get('TP2_ROOT', Path.cwd())).resolve()
assert (ROOT/'src/lib').is_dir(), 'Indicar TP2_ROOT con la raíz del repositorio'
os.chdir(ROOT)
sys.path.insert(0,str(ROOT/'src'))
print(ROOT)

## 2. Instalar dependencias

Colab trae PyTorch y torchvision compatibles con CUDA, como indica la clase 2. Instalar solo las dependencias faltantes. En local usar el requirements del proyecto.

In [ ]:
import importlib.util
import subprocess
paquetes = {'pydantic_settings':'pydantic-settings','dotenv':'python-dotenv','kagglehub':'kagglehub',
            'ultralytics':'ultralytics','albumentations':'albumentations','onnxruntime':'onnxruntime'}
faltantes = [p for mod,p in paquetes.items() if importlib.util.find_spec(mod) is None]
if faltantes:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*faltantes])

## 3. Configuración del entorno

La semilla controla muestreo e inicialización. La base JSON en Colab evita depender de PostgreSQL; el entrenamiento no necesita indexar allí. El tamaño y la dimensión son compartidos con la app.

In [ ]:
os.environ['USE_PGVECTOR']='false'
# Los paths del .env local son relativos a src; los notebooks se ejecutan desde la raíz.
from dotenv import dotenv_values
local_env = ROOT / 'src' / '.env'
if local_env.is_file():
    path_keys = {'DATA_PATH', 'DATASET_PATH', 'OUTPUT_PATH', 'MODEL_PATH', 'EMBEDDINGS_PATH', 'TORCH_HOME', 'YOLO_MODEL'}
    for key, value in dotenv_values(local_env).items():
        if value is not None:
            os.environ.setdefault(key, str((local_env.parent / value).resolve()) if key in path_keys else value)

from lib.config import settings
from lib.tp2_data import fijar_semilla
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import cv2
from PIL import Image
from IPython.display import display
SEED=42
fijar_semilla(SEED)
DATA=Path(settings.dataset_path)
IMAGE_SIZE=settings.image_size
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dataset:',DATA,'| dispositivo:',DEVICE,'| PyTorch:',torch.__version__)
print('IMAGE_SIZE:',IMAGE_SIZE,'| EMBEDDING_DIM:',settings.embedding_dim)
assert settings.embedding_dim==512, 'Se eligió ResNet18 y CNN de 512 dimensiones'

## 4. Dataset y splits

Se usan los archivos de 1-pre-processing sin recalcular el split. Se controla que train, valid, test e independiente no compartan rutas. Se exige el conjunto independiente antes de presentar los resultados finales.

In [ ]:
from lib.tp2_data import cargar_splits, tensor_bgr
from lib.tp2_retrieval import features
from lib.tp2_retrieval import baseline, extraer_matriz, evaluar_busqueda, resumen_busqueda, calibrar
from lib.evaluation.metrics import ndcg_at_k
from sklearn.decomposition import PCA
import json
import time
from torch import nn
if not all((DATA/s).is_dir() for s in ['train','valid','test']):
    subprocess.check_call([sys.executable,'scripts/download_dataset.py'])
splits=cargar_splits(DATA)
filas={s:splits[splits.split==s].reset_index(drop=True) for s in ['train','valid','test','independent']}
CLASES=sorted(filas['train'].breed.unique())
assert len(CLASES)==70 and set(CLASES)==set((ROOT/'razas_dataset.txt').read_text(encoding='utf-8-sig').splitlines())
assert all(len(filas[s]) for s in ['train','valid','test'])
display(splits.groupby('split').agg(imagenes=('path','size'),razas=('breed','nunique')))
if filas['independent'].empty: raise ValueError('Falta el split independiente en el manifiesto')
OUT=Path(settings.output_path); OUT.mkdir(exist_ok=True,parents=True)
MODELS=Path(settings.model_path); MODELS.mkdir(exist_ok=True,parents=True)
CP_A=MODELS/settings.resnet18_model_name; CP_B=MODELS/settings.cnn_custom_model_name
BATCH=settings.train_batch_size
print('Clases:',len(CLASES))

## 5. Evaluación de búsqueda por similitud
ResNet18 con pesos ImageNet produce un vector de 512 componentes tras el pooling global. Con vectores unitarios, la distancia L2 y el coseno ordenan los vecinos de la misma forma. Cada vecino suma su similitud a su raza. El score devuelto es la media de la raza ganadora y el rechazo depende del mejor vecino. En validación se busca la mayor accuracy entre consultas aceptadas conservando al menos el 90 % de cobertura. Este presupuesto limita a una de cada diez las consultas conocidas que pueden rechazarse. La tasa de falsas aceptaciones de razas desconocidas requiere ejemplos de esas razas. NDCG@10 utiliza la cantidad de imágenes relevantes de toda la galería para calcular el ranking ideal.

In [ ]:
def evaluar_modelo_embeddings(model,nombre):
    gal=extraer_matriz(model,DATA,filas['train'],CLASES,IMAGE_SIZE,BATCH)
    consultas={s:extraer_matriz(model,DATA,filas[s],CLASES,IMAGE_SIZE,BATCH) for s in ['valid','test','independent'] if len(filas[s])}
    print(nombre,'| galería:',gal.shape)
    assert np.allclose(np.linalg.norm(gal,axis=1),1,atol=1e-4)
    registros=[]; detalles={}; configs=[]
    for metrica in ['cosine','l2']:
        tabla,best=calibrar(gal,filas['train'],consultas['valid'],filas['valid'],metrica)
        display(tabla.head(10))
        tabla.to_csv(OUT/f'{nombre}_{metrica}_calibracion.csv',index=False)
        configs.append({'modelo':nombre,'metric':metrica,**best})
        # Mantener TOP-10 para NDCG/precision; k seleccionado solo controla voto.
        for split,q in consultas.items():
            det=evaluar_busqueda(gal,filas['train'],q,filas[split],metrica,int(best['k']),best['threshold'])
            # Comprobar contra helper de cátedra en un ejemplo.
            first=det.iloc[0]; relev=(filas['train'].iloc[first.neighbors].breed==first.breed).astype(int).tolist()
            nrel=int((filas['train'].breed==first.breed).sum())
            assert np.isclose(ndcg_at_k(relev,k=10,n_relevant=nrel),first.ndcg)
            det10=evaluar_busqueda(gal,filas['train'],q,filas[split],metrica,10,best['threshold']) if int(best['k'])!=10 else det
            registros.append({'modelo':nombre,'metric':metrica,'split':split,'k_voto':int(best['k']),
                              **resumen_busqueda(det),'accuracy_voto@10':det10.correct.mean()})
            detalles[(metrica,split)]=det
            if split!='valid': display(det.groupby('breed')[['ndcg','precision10','correct']].mean().sort_values('ndcg').head(10))
    return pd.DataFrame(registros),detalles,pd.DataFrame(configs),consultas['test']

base=baseline(DEVICE)
r_base,d_base,c_base,e_base=evaluar_modelo_embeddings(base,'baseline')
display(r_base)
del base
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 5.1. Ejemplos de recuperación y análisis de TOP-K

Visualizar una consulta con NDCG alto y una con NDCG bajo, coloreando aciertos. La forma de la curva K/umbral se analiza en validación; no elegir K por el test.

In [ ]:
def mostrar_vecinos(det,indice):
    row=det.iloc[indice]; fig,axs=plt.subplots(2,6,figsize=(15,5))
    for ax in axs.flat: ax.axis('off')
    axs.flat[0].imshow(Image.open(DATA/row.path).convert('RGB')); axs.flat[0].set_title(f'Consulta\n{row.breed}',fontsize=8)
    for ax,idx,score in zip(axs.flat[1:],row.neighbors,row.neighbor_scores):
        v=filas['train'].iloc[idx]; ax.imshow(Image.open(DATA/v.path).convert('RGB'))
        ax.set_title(f'{v.breed}\n{score:.3f}',color='green' if v.breed==row.breed else 'red',fontsize=8)
    plt.suptitle(f'Pred: {row.pred} | NDCG@10={row.ndcg:.3f}'); plt.tight_layout(); plt.show()
for split in ['test','independent']:
    if ('cosine',split) not in d_base: continue
    det=d_base[('cosine',split)]
    mostrar_vecinos(det,int(det.ndcg.argmax())); mostrar_vecinos(det,int(det.ndcg.argmin()))
for met in ['cosine','l2']:
    tabla=pd.read_csv(OUT/f'baseline_{met}_calibracion.csv')
    fig,ax=plt.subplots(figsize=(7,4))
    for k,g in tabla.groupby('k'): ax.plot(g.sort_values('threshold').threshold,g.sort_values('threshold').accuracy,'o-',label=f'K={k}')
    ax.set(xlabel='Umbral',ylabel='Accuracy valid',title=met); ax.legend(); plt.show()

## 6. Modelo A. ResNet18
Se comparan el backbone congelado y el ajuste completo. La cabeza aleatoria usa una tasa inicial de 0,001 y el backbone preentrenado usa 0,0001 para conservar sus características. AdamW separa la regularización de la actualización por gradiente. El scheduler coseno reduce la tasa a lo largo de las épocas. La entropía cruzada usa label smoothing de 0,1. Cada experimento conserva el checkpoint de mayor accuracy de validación.

In [ ]:
from lib.bootstrap import build_classifier
import shutil
classifier=build_classifier(settings)
classifier.set_active_model('resnet18_finetuned')
# Dos estrategias, guardadas aparte para no sobrescribir la candidata anterior.
experimentos=[]
for estrategia in ['frozen','full']:
    classifier.train_config['strategy']=estrategia
    classifier.train_classifier()
    hist=pd.DataFrame(json.loads((OUT/'resnet18_finetuned_history.json').read_text())['history'])
    hist.to_json(OUT/f'resnet18_finetuned_{estrategia}_hist.json',orient='records')
    candidato=MODELS/f'resnet18_{estrategia}_candidato.pth'
    shutil.copy2(CP_A,candidato)
    experimentos.append({'modelo':'resnet18_finetuned','estrategia':estrategia,'val_acc':hist.val_acc.max(),'epocas':len(hist),'checkpoint':str(candidato)})
exp_a=pd.DataFrame(experimentos); display(exp_a)
mejor_a=exp_a.sort_values('val_acc',ascending=False).iloc[0]
shutil.copy2(mejor_a.checkpoint,CP_A)
classifier.train_config["strategy"]=mejor_a.estrategia
print('Estrategia elegida por validación:',mejor_a.estrategia)


## 6.1. Curvas y confusión de ResNet

Accuracy y loss de train/valid: caída train y aumento valid sugieren overfitting; ambos malos sugieren underfitting. La matriz se normaliza por fila (clase real), por lo tanto la diagonal es recall. Mostrar pares con más confusiones en cantidad y tabla de peor recall/F1 con soporte.

In [ ]:
def mostrar_curvas(hist,titulo):
    fig,axs=plt.subplots(1,2,figsize=(12,4))
    for ax,m in zip(axs,['loss','acc']):
        ax.plot(hist.epoch,hist[f'train_{m}'],label='train'); ax.plot(hist.epoch,hist[f'val_{m}'],label='valid'); ax.set(xlabel='Época',title=f'{titulo} - {m}'); ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

def mostrar_evaluacion(resultado,titulo):
    resumen,tabla,cm,pred=resultado
    display(pd.DataFrame([resumen])); display(tabla[tabla.support>0].sort_values('f1').head(10))
    norm=np.divide(cm,cm.sum(1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(1,keepdims=True)!=0)
    fig,ax=plt.subplots(figsize=(15,13)); im=ax.imshow(norm,cmap='Blues',vmin=0,vmax=1); fig.colorbar(im,ax=ax)
    ax.set_xticks(range(len(CLASES)),CLASES,rotation=90,fontsize=6); ax.set_yticks(range(len(CLASES)),CLASES,fontsize=6)
    ax.set(xlabel='Predicción',ylabel='Real',title=titulo); plt.tight_layout(); plt.show()
    errores=cm.copy(); np.fill_diagonal(errores,0)
    pares=[(CLASES[i],CLASES[j],errores[i,j]) for i,j in zip(*np.where(errores>0))]
    display(pd.DataFrame(sorted(pares,key=lambda x:-x[2])[:10],columns=['real','pred','cantidad']))
for estrategia in ['frozen','full']:
    mostrar_curvas(pd.read_json(OUT/f'resnet18_finetuned_{estrategia}_hist.json'),estrategia)
def evaluar_modelo(nombre, split):
    classifier.set_active_model(nombre)
    resumen = classifier.evaluate_classifier(split)
    detalle = classifier.last_eval
    return resumen, detalle['table'], detalle['confusion_matrix'], detalle['predictions']

a_test=evaluar_modelo('resnet18_finetuned','test')
mostrar_evaluacion(a_test,'ResNet18 - test')

## 7. Modelo B (obligatorio): CNN propia

Inspirada en CNN base de clase 1 y normalización de clase 2. Cuatro bloques Conv3×3 con padding 1 → BatchNorm → ReLU → MaxPool2×2. Canales 32,64,128,256; H/W 224→112→56→28→14. GAP reduce 256×14×14 a 256, embedding lineal 512 y Dropout antes de la cabeza (reduce coadaptación durante entrenamiento). Una capa lineal final produce logits por raza; no softmax durante CE. Comparar dos dropout (.3 y .5) manteniendo lo demás fijo: no afirmar que .5 mejora sin evidencia. A diferencia de ResNet, todos los parámetros empiezan aleatorios. Entrenamiento y selección por validación como en A.

In [ ]:
from lib.models.cnn_custom import CNNCustom
cnn=CNNCustom(len(CLASES),settings.embedding_dim)
print(cnn)
with torch.no_grad():
    cnn.eval(); x=torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE)
    print('Mapas:',cnn.features(x).shape,'embedding:',cnn.forward_features(x).shape,'logits:',cnn(x).shape)
print('Parámetros:',sum(p.numel() for p in cnn.parameters()))
del cnn
classifier.set_active_model('cnn_custom')

experimentos_b=[]
for dropout in [.3,.5]:
    classifier.train_config['dropout']=dropout
    classifier.train_classifier()
    hist=pd.DataFrame(json.loads((OUT/'cnn_custom_history.json').read_text())['history'])
    candidato=MODELS/f'cnn_dropout_{dropout}_candidato.pth'
    shutil.copy2(CP_B,candidato)
    hist.to_json(OUT/f'cnn_dropout_{dropout}_hist.json',orient='records',indent=2)
    experimentos_b.append({'modelo':'cnn_custom','dropout':dropout,'val_acc':hist.val_acc.max(),'epocas':len(hist),'checkpoint':str(candidato)})
exp_b=pd.DataFrame(experimentos_b); display(exp_b)
mejor_b=exp_b.sort_values('val_acc',ascending=False).iloc[0]
shutil.copy2(mejor_b.checkpoint,CP_B)
settings.cnn_dropout=float(mejor_b.dropout)


## 7.1. Curvas y confusión de CNN propia

Comparar ambos experimentos con los mismos splits. Evaluar en test únicamente el checkpoint seleccionado por validación. Si CNN rinde peor, comprobar curvas, errores por raza y capacidad antes de atribuirlo solo a ausencia de preentrenamiento.

In [ ]:
for dropout in [.3,.5]: mostrar_curvas(pd.read_json(OUT/f'cnn_dropout_{dropout}_hist.json'),f'CNN dropout {dropout}')
b_test=evaluar_modelo('cnn_custom','test')
mostrar_evaluacion(b_test,'CNN propia - test')

## 8. Evaluación sobre el conjunto independiente

Mismo preprocesamiento sin augmentation ni ajuste de hiperparámetros. Macro promedio sobre razas con soporte real en el split (en test están las 70; independiente puede tener menos). Informar cobertura de razas y comparar con test. No afirmar que una caída es necesariamente por encuadre: revisar ejemplos, diferencias de composición y soporte.

In [ ]:
resultados_cls={('resnet18_finetuned','test'):a_test,('cnn_custom','test'):b_test}
if len(filas['independent']):
    for name,path in [('resnet18_finetuned',CP_A),('cnn_custom',CP_B)]:
        r=evaluar_modelo(name,'independent')
        resultados_cls[(name,'independent')]=r
        mostrar_evaluacion(r,name+' - independiente')
else: print('PENDIENTE: no hay consultas independientes etiquetadas.')
tabla_cls=pd.DataFrame([{'modelo':m,'split':s,**r[0]} for (m,s),r in resultados_cls.items()]); display(tabla_cls)

## 9. Embeddings de los modelos entrenados

Misma galería y consultas que baseline. ResNet extrae avgpool antes de fc; CNN usa su capa de embedding antes de dropout y clasificador. L2 normaliza, PCA de 2 componentes por separado con el mismo subconjunto de razas; la proyección no demuestra por sí sola mayor NDCG. Comparar cuantitativamente primero.

In [ ]:
r_todos=[r_base]; c_todos=[c_base]; d_todos={'baseline':d_base}; e_todos={'baseline':e_base}
for name,path in [('resnet18_finetuned',CP_A),('cnn_custom',CP_B)]:
    model=torch.load(path,map_location=DEVICE,weights_only=False).eval()
    r,d,c,e=evaluar_modelo_embeddings(model,name)
    r_todos.append(r); c_todos.append(c); d_todos[name]=d; e_todos[name]=e
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
tabla_retrieval=pd.concat(r_todos,ignore_index=True); tabla_calibracion=pd.concat(c_todos,ignore_index=True)
display(tabla_retrieval)
razas=CLASES[:8]; mask=filas['test'].breed.isin(razas).to_numpy()
fig,axs=plt.subplots(1,3,figsize=(16,5))
for ax,(name,e) in zip(axs,e_todos.items()):
    pca=PCA(2); z=pca.fit_transform(e[mask]); labels=filas['test'].breed.to_numpy()[mask]
    for raza in razas:
        m=labels==raza; ax.scatter(z[m,0],z[m,1],s=15,label=raza,alpha=.7)
    ax.set_title(f'{name}\nVarianza explicada {pca.explained_variance_ratio_.sum():.1%}')
axs[-1].legend(fontsize=7,bbox_to_anchor=(1,1)); plt.tight_layout(); plt.show()

## 10. Análisis de errores

En multiclase cada error es simultáneamente FN para raza real y FP para raza predicha. Mostrar errores de alta confianza y los pares más frecuentes, con hipótesis observables (pelaje, pose, fondo, oclusión) en Markdown tras ejecutar. Un score softmax no es probabilidad calibrada de que la predicción sea correcta.

In [ ]:
for (name,split),r in resultados_cls.items():
    pred=r[3]; err=pred[pred.real!=pred.pred].sort_values('score',ascending=False).head(8)
    print(name,split,'| errores:',int((pred.real!=pred.pred).sum()))
    if err.empty: continue
    fig,axs=plt.subplots(2,4,figsize=(14,7))
    for ax in axs.flat: ax.axis('off')
    for ax,row in zip(axs.flat,err.itertuples()):
        ax.imshow(Image.open(DATA/row.path).convert('RGB')); ax.set_title(f'Real: {CLASES[row.real]}\nPred: {CLASES[row.pred]} ({row.score:.2f})',fontsize=9,color='red')
    plt.tight_layout(); plt.show()
for name,detalles in d_todos.items():
    if name=='baseline': continue
    det=detalles[('cosine','test')]
    print(name); mostrar_vecinos(det,int(det.ndcg.argmin()))

## 11. Costo computacional

Clase 7: comparar CPU bajo el mismo protocolo; 10 warmups y 50 inferencias, batch=1, torch.no_grad y eval, misma cantidad de hilos, misma entrada preprocesada. Medimos forward puro (sin lectura/resize); media y P95. Tamaño en MB del archivo completo en disco, no VRAM. Baseline sin checkpoint local usa tamaño de parámetros y buffers como estimación y se identifica como tal.

In [ ]:
def costo(model,path=None):
    model=model.cpu().eval(); x=torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE)
    tiempos=[]
    with torch.no_grad():
        for _ in range(10): model(x)
        for _ in range(50):
            t=time.perf_counter(); model(x); tiempos.append((time.perf_counter()-t)*1000)
        dim=features(model,x).shape[1]
    bytes_tensor=sum(p.numel()*p.element_size() for p in list(model.parameters())+list(model.buffers()))
    return {'parametros':sum(p.numel() for p in model.parameters()),'MB':(path.stat().st_size if path else bytes_tensor)/1e6,
            'tipo_MB':'checkpoint' if path else 'tensores estimados','CPU_ms':np.mean(tiempos),'P95_ms':np.percentile(tiempos,95),'embedding_dim':dim}
print('Hilos CPU:',torch.get_num_threads())
costos=[{'modelo':'baseline',**costo(baseline(torch.device('cpu')))}]
for name,path in [('resnet18_finetuned',CP_A),('cnn_custom',CP_B)]:
    costos.append({'modelo':name,**costo(torch.load(path,map_location='cpu',weights_only=False),path)})
tabla_costo=pd.DataFrame(costos); display(tabla_costo)

## 12. Comparación y elección final

Se reportan test e independiente sin usarlos para seleccionar. Criterio inicial: clasificador con mayor accuracy de validación (empate: menor latencia CPU); retrieval con mayor NDCG@10 valid (empate: accuracy y costo). Comparar cosine/l2, eligiendo coseno ante igualdad porque pgvector ordena así. Ajustar solo con valid; documentar si la aplicación prioriza latencia u otro requisito antes de evaluar test. Similitud devuelve ejemplos y permite agregar referencias sin reentrenar; clasificación supervisada produce predicción rápida pero depende de clases fijas y puede tener problemas fuera de dominio.

In [ ]:
display(tabla_cls); display(tabla_retrieval); display(tabla_costo)
seleccion_cls=pd.DataFrame([{'modelo':'resnet18_finetuned','val_acc':mejor_a.val_acc},{'modelo':'cnn_custom','val_acc':mejor_b.val_acc}]).merge(tabla_costo,on='modelo')
mejor_cls=seleccion_cls.sort_values(['val_acc','CPU_ms'],ascending=[False,True]).iloc[0]
seleccion_retr=tabla_calibracion.merge(tabla_costo[['modelo','CPU_ms']],on='modelo')
mejor_retr=seleccion_retr.sort_values(['ndcg','accuracy','CPU_ms','metric'],ascending=[False,False,True,True]).iloc[0]
print('Clasificador elegido por valid:',mejor_cls.modelo)
print('Embeddings elegidos por valid:',mejor_retr.modelo)
config_final={'EMBEDDING_MODEL':mejor_retr.modelo,'EMBEDDING_DIM':512,'SIMILARITY_METRIC':mejor_retr.metric,
              'SIMILARITY_THRESHOLD':float(mejor_retr.threshold),'TOP_K':int(mejor_retr.k),
              'CLASSIFIER_MODEL':mejor_cls.modelo}
print(json.dumps(config_final,indent=2)); (OUT/'config_recomendada.json').write_text(json.dumps(config_final,indent=2))
for nombre,t in [('clasificacion',tabla_cls),('retrieval',tabla_retrieval),('costo',tabla_costo)]: t.to_csv(OUT/f'tp2_{nombre}.csv',index=False)


## 13. Exportación de checkpoints

Se usa torch.save(model,path), no state_dict. CNNCustom vive en lib.models.cnn_custom y la lista class_names se guarda en el objeto. No cargar archivos .pth no confiables. Copiar los checkpoints finales a models/ del repositorio local y publicar links de solo lectura, nunca commitear pesos. Si Colab: descargar modelos y también splits_finales.csv y resultados requeridos. La exportación no requiere ONNX para este TP.

In [ ]:
for path in [CP_A,CP_B]:
    assert path.is_file(); print(path,path.stat().st_size/1e6,'MB')
# En Colab: ejecutar si se quiere descargar.
# from google.colab import files
# files.download(str(CP_A)); files.download(str(CP_B))

## 14. Etapa 3 — Validación YOLO y clasificación de recortes

Clase 2 y 7: YOLO preentrenado COCO; la clase dog se filtra según configuración, sin entrenamiento del detector. Se prueban tres umbrales (0.15/0.25/0.50) sobre escenas independientes. Más alto suele reducir FP y también perder perros difíciles. Sin anotaciones de cajas, solo comparación visual y de conteo; no reportar mAP. La clasificación de recortes usa el modelo elegido por valid, mismo tensor BGR→RGB/resize/normalización, no clasificamos la escena completa si hay varios perros. La elección de umbral debe justificarse con ejemplos de validación separados de la prueba final; si solo hay escenas independientes, reportar sensibilidad sin afirmar evaluación imparcial de un umbral ajustado allí.

In [ ]:
from lib.bootstrap import build_detection
from lib.visualization.draw import draw_detections
p=DATA/'independent_scenes.csv'
if p.is_file():
    escenas=pd.read_csv(p,dtype=str,keep_default_na=False)
    limpias=filas['independent'][['path','breed']].head(2).assign(kind='clean',expected_dogs=1)
    escenas=pd.concat([limpias,escenas[escenas.kind!='clean']],ignore_index=True)
else: escenas=filas['independent'][['path','breed']].head(2).assign(kind='clean',expected_dogs=1)
if not p.is_file(): print('PENDIENTE: independent_scenes.csv con escenas complejas y sin perro')
if len(escenas):
    classifier.set_active_model(mejor_cls.modelo)
    detector = build_detection(settings, classifier)
    filas_det=[]
    for row in escenas.itertuples():
        image=cv2.imread(str(DATA/row.path)); assert image is not None
        h,w=image.shape[:2]
        fig,axs=plt.subplots(1,3,figsize=(15,5))
        for ax,threshold in zip(axs,[.15,.25,.50]):
            detector.conf_threshold = threshold
            detecciones = []
            for (x1,y1,x2,y2), score in detector.detect_dogs(image):
                breed, confidence = detector.classify_detected_dog(image[y1:y2,x1:x2])
                detecciones.append(dict(bbox=[x1,y1,x2,y2], breed=breed, breed_score=confidence, det_score=score))
            canvas = draw_detections(image, detecciones)
            count = len(detecciones)
            ax.imshow(cv2.cvtColor(canvas,cv2.COLOR_BGR2RGB)); ax.set_title(f'conf={threshold} | {count} perros'); ax.axis('off')
            filas_det.append({'path':row.path,'kind':row.kind,'threshold':threshold,'count':count,'expected_dogs':getattr(row,'expected_dogs','')})
        plt.tight_layout(); plt.show()
    display(pd.DataFrame(filas_det)); pd.DataFrame(filas_det).to_csv(OUT/'yolo_thresholds.csv',index=False)
else: print('PENDIENTE: agregar escenas con múltiples perros, complejas y sin perros.')